In [ ]:
"""
产品级别数据整理脚本
从原始交易数据生成：
1. 企业-产品-年份级别的产出和外包数据
2. 产品特征数据
"""

import pandas as pd
import numpy as np

print("=" * 80)
print("产品级别数据整理")
print("=" * 80)

# ============================================================================
# 步骤1：读取原始交易数据
# ============================================================================
print("\n[1] 读取原始交易数据...")

# 请替换为你的实际数据文件路径
DATA_PATH = r'G:\Kuangyu_Temp\Outsource\lenth9.dta'  # 或 .csv

try:
    if DATA_PATH.endswith('.dta'):
        df = pd.read_stata(DATA_PATH)
    elif DATA_PATH.endswith('.csv'):
        df = pd.read_csv(DATA_PATH)
    else:
        raise ValueError("不支持的文件格式")
   
    print(f"✓ 数据读取成功")
    print(f"  原始记录数: {len(df):,}")
    print(f"  列名: {list(df.columns)}")
except Exception as e:
    print(f"✗ 数据读取失败: {e}")
    exit(1)

# ============================================================================
# 步骤2：数据清洗和准备
# ============================================================================
print("\n[2] 数据清洗...")

# 确保只使用year列（如果存在date列，提取year）
if 'date' in df.columns and 'year' not in df.columns:
    print("  从date列提取year...")
    df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
    # 如果date是字符串格式如"2017-01"，提取前4位
    if df['year'].isna().all():
        df['year'] = df['date'].astype(str).str[:4].astype(int, errors='ignore')

# 检查必需列
required_cols = ['year', 'firm_id', 'product_id', 'is_output', 'v']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    print(f"✗ 缺少必需列: {missing_cols}")
    exit(1)

# 删除缺失值
original_len = len(df)
df = df.dropna(subset=required_cols)
print(f"  删除缺失值: {original_len - len(df):,} 行")

# 删除负值交易
negative_count = (df['v'] < 0).sum()
if negative_count > 0:
    print(f"  删除负值交易: {negative_count:,} 行")
    df = df[df['v'] >= 0]

print(f"✓ 清洗后记录数: {len(df):,}")

# ============================================================================
# 步骤3：计算企业-产品-年份级别的产出和投入
# ============================================================================
print("\n[3] 计算企业-产品-年份级别的汇总数据...")

# 分别计算产出和投入
output_df = df[df['is_output'] == 1].groupby(['year', 'firm_id', 'product_id'])['v'].sum().reset_index()
output_df.columns = ['year', 'firm_id', 'product_id', 'total_output']

input_df = df[df['is_output'] == 0].groupby(['year', 'firm_id', 'product_id'])['v'].sum().reset_index()
input_df.columns = ['year', 'firm_id', 'product_id', 'total_input']

print(f"  产出记录: {len(output_df):,}")
print(f"  投入记录: {len(input_df):,}")

# ============================================================================
# 步骤4：合并产出和投入，计算外包值和自产值
# ============================================================================
print("\n[4] 计算外包值和自产值...")

# 合并产出和投入（以产出为主）
product_level = output_df.merge(
    input_df,
    on=['year', 'firm_id', 'product_id'],
    how='left'
)

# 填充没有投入的产品（全部自产）
product_level['total_input'] = product_level['total_input'].fillna(0)

# 计算外包值：min(投入, 产出)
product_level['outsourcing_value'] = np.minimum(
    product_level['total_input'],
    product_level['total_output']
)

# 计算自产值：产出 - 外包
product_level['production_value'] = (
    product_level['total_output'] - product_level['outsourcing_value']
)

# 计算外包比例
product_level['outsourcing_percen'] = (
    product_level['outsourcing_value'] / product_level['total_output']
)

# 处理除以0的情况
product_level['outsourcing_percen'] = product_level['outsourcing_percen'].fillna(0)

print(f"✓ 企业-产品-年份级别数据: {len(product_level):,} 条")
print(f"\n描述性统计:")
print(product_level[['total_output', 'total_input', 'outsourcing_value',
                     'production_value', 'outsourcing_percen']].describe())

# ============================================================================
# 步骤5：保存第一份表（企业-产品-年份级别）
# ============================================================================
print("\n[5] 保存企业-产品-年份级别数据...")

# 选择并重命名列
firm_product_table = product_level[[
    'year', 'firm_id', 'product_id',
    'total_output', 'outsourcing_value', 'production_value',
    'outsourcing_percen'
]].copy()

# 保存为Stata格式
OUTPUT_FILE_1 = 'firm_product_year_level.dta'
firm_product_table.to_stata(OUTPUT_FILE_1, write_index=False)
print(f"✓ 已保存: {OUTPUT_FILE_1}")
print(f"  包含 {len(firm_product_table):,} 条记录")


# ============================================================================
# 步骤6：计算产品特征数据（不分年份，汇总所有年份）
# ============================================================================
print("\n[6] 计算产品特征数据（汇总所有年份）...")

# 按product_id汇总，不分年份
product_characteristics = product_level.groupby('product_id').agg({
    'firm_id': 'nunique',  # 生产该产品的企业数（跨年份去重）
    'total_output': 'sum',  # 总产出（所有年份）
    'outsourcing_value': 'sum',  # 总外包（所有年份）
    'production_value': 'sum',  # 总自产（所有年份）
    'year': lambda x: list(x.unique())  # 该产品出现的年份
}).reset_index()

# 重命名列
product_characteristics.columns = [
    'product_id',
    'num_firms',  # 生产该产品的企业数（popularity）
    'total_output',
    'total_outsourcing',
    'total_production',
    'years_present'  # 该产品出现的年份列表
]

# 计算产品出现的年份数
product_characteristics['num_years'] = product_characteristics['years_present'].apply(len)

# 删除years_present列（如果不需要的话）
product_characteristics = product_characteristics.drop('years_present', axis=1)

# 计算总外包强度
product_characteristics['outsourcing_intensity'] = (
    product_characteristics['total_outsourcing'] /
    product_characteristics['total_output']
)

# 处理除以0
product_characteristics['outsourcing_intensity'] = (
    product_characteristics['outsourcing_intensity'].fillna(0)
)

# 添加其他有用的特征
# 平均每家企业的产出
product_characteristics['avg_output_per_firm'] = (
    product_characteristics['total_output'] /
    product_characteristics['num_firms']
)

# 平均每年的产出
product_characteristics['avg_output_per_year'] = (
    product_characteristics['total_output'] /
    product_characteristics['num_years']
)

# 计算产品的总产出排名（越小越重要）
product_characteristics['output_rank'] = (
    product_characteristics['total_output']
    .rank(ascending=False, method='dense')
    .astype(int)
)

print(f"✓ 产品特征数据: {len(product_characteristics):,} 个产品")
print(f"\n产品受欢迎程度分布:")
print(product_characteristics['num_firms'].describe())

print(f"\n外包强度分布:")
print(product_characteristics['outsourcing_intensity'].describe())

print(f"\n产品出现年份数分布:")
print(product_characteristics['num_years'].value_counts().sort_index())

# ============================================================================
# 步骤7：保存第二份表（产品特征）
# ============================================================================
print("\n[7] 保存产品特征数据...")

# 按产品受欢迎程度排序
product_characteristics = product_characteristics.sort_values(
    'num_firms',
    ascending=False
)

# 保存为Stata格式
OUTPUT_FILE_2 = 'product_characteristics.dta'
product_characteristics.to_stata(OUTPUT_FILE_2, write_index=False)
print(f"✓ 已保存: {OUTPUT_FILE_2}")
print(f"  包含 {len(product_characteristics):,} 个产品")



# ============================================================================
# 步骤8：生成汇总报告
# ============================================================================
print("\n" + "=" * 80)
print("数据整理完成！")
print("=" * 80)

print("\n【输出文件】")
print(f"1. firm_product_year_level.dta / .csv")
print(f"   - 企业-产品-年份级别数据")
print(f"   - {len(firm_product_table):,} 条记录")
print(f"   - 列: year, firm_id, product_id, total_output,")
print(f"         outsourcing_value, production_value, outsourcing_percen")

print(f"\n2. product_characteristics.dta / .csv")
print(f"   - 产品特征数据（不分年份）")
print(f"   - {len(product_characteristics):,} 个产品")
print(f"   - 列: product_id, num_firms, total_output,")
print(f"         total_outsourcing, total_production, outsourcing_intensity,")
print(f"         avg_output_per_firm, avg_output_per_year, output_rank, num_years")

print("\n【关键统计】")
print(f"年份范围: {product_level['year'].min()} - {product_level['year'].max()}")
print(f"企业数: {product_level['firm_id'].nunique():,}")
print(f"产品数: {product_level['product_id'].nunique():,}")
print(f"企业-产品-年份记录数: {len(product_level):,}")

print("\n【前10个最受欢迎的产品】")
top_products = product_characteristics.nlargest(10, 'num_firms')[
    ['product_id', 'num_firms', 'total_output', 'outsourcing_intensity', 'num_years']
]
print(top_products.to_string(index=False))

print("\n【外包强度最高的10个产品（至少10家企业生产）】")
high_outsourcing = product_characteristics[
    product_characteristics['num_firms'] >= 10
].nlargest(10, 'outsourcing_intensity')[
    ['product_id', 'num_firms', 'outsourcing_intensity', 'num_years']
]
print(high_outsourcing.to_string(index=False))

print("\n" + "=" * 80)
print("所有数据已成功整理！")
print("=" * 80)

产品级别数据整理

[1] 读取原始交易数据...
✓ 数据读取成功
  原始记录数: 465,487,031
  列名: ['firm_id', 'product_id', 'is_output', 'year', 'v']

[2] 数据清洗...
  删除缺失值: 0 行
✓ 清洗后记录数: 465,487,031

[3] 计算企业-产品-年份级别的汇总数据...
  产出记录: 90,296,650
  投入记录: 375,190,381

[4] 计算外包值和自产值...
✓ 企业-产品-年份级别数据: 90,296,650 条

描述性统计:
       total_output   total_input  outsourcing_value  production_value  \
count  9.029665e+07  9.029665e+07       9.029665e+07      9.029665e+07   
mean   4.643103e+06  2.638616e+06       1.669839e+06      2.973264e+06   
std    2.382010e+08  1.688639e+08       1.454608e+08      1.644814e+08   
min    1.110223e-16  0.000000e+00       0.000000e+00      0.000000e+00   
25%    2.079000e+03  0.000000e+00       0.000000e+00      2.000000e+02   
50%    2.000000e+04  0.000000e+00       0.000000e+00      6.296000e+03   
75%    2.298000e+05  2.165694e+04       6.999000e+03      1.131780e+05   
max    5.511644e+11  3.657895e+11       3.388452e+11      5.511644e+11   

       outsourcing_percen  
count        9.029665e+

In [6]:
import pandas as pd
import os
os.chdir(r'G:\Kuangyu_Temp\Outsource')
# 输入: firm_product_year_level.dta
# 输出: product_characteristics.dta

df = pd.read_stata("firm_product_year_level.dta")
df["year"] = df["year"].astype(int)

# 企业数跨年份去重
num_firms = (
    df[["product_id", "firm_id"]]
    .drop_duplicates()
    .groupby("product_id", as_index=False)
    .size()
    .rename(columns={"size": "num_firms"})
)

# 外包企业数跨年份去重（outsourcing_value > 0）
num_firms_outsourcing = (
    df[df["outsourcing_value"] > 0][["product_id", "firm_id"]]
    .drop_duplicates()
    .groupby("product_id", as_index=False)
    .size()
    .rename(columns={"size": "num_firms_outsourcing"})
)

product_chars = df.groupby("product_id", as_index=False).agg(
    total_output      = ("total_output",      "sum"),
    total_outsourcing = ("outsourcing_value",  "sum"),
    total_production  = ("production_value",   "sum"),
    num_years         = ("year",               "nunique"),
)

product_chars = product_chars.merge(num_firms, on="product_id", how="left")
product_chars = product_chars.merge(num_firms_outsourcing, on="product_id", how="left")
product_chars["num_firms_outsourcing"] = product_chars["num_firms_outsourcing"].fillna(0).astype(int)

product_chars["outsourcing_intensity"] = (
    product_chars["total_outsourcing"] / product_chars["total_output"]
).fillna(0)

product_chars["avg_output_per_firm"] = (
    product_chars["total_output"] / product_chars["num_firms"]
)
product_chars["avg_output_per_year"] = (
    product_chars["total_output"] / product_chars["num_years"]
)
product_chars["pct_firms_outsourcing"] = (
    product_chars["num_firms_outsourcing"] / product_chars["num_firms"]
).fillna(0)

product_chars = product_chars.sort_values("num_firms", ascending=False).reset_index(drop=True)

product_chars.to_stata("product_characteristics.dta", write_index=False)

In [ ]:
import pandas as pd
import numpy as np
import os
os.chdir(r'G:\Kuangyu_Temp\Outsource')



df = pd.read_stata("firm_product_year_level.dta")
df["year"] = df["year"].astype(int)

# ============================================================
# 公司-年份层面汇总
# ============================================================
firm_summary = df.groupby(["year", "firm_id"]).agg(
    firm_total_output    = ("total_output",      "sum"),
    firm_total_outsource = ("outsourcing_value", "sum"),
    n_products           = ("product_id",        "count"),
).reset_index()

firm_summary["outsourcing_intensity"] = (
    firm_summary["firm_total_outsource"] / firm_summary["firm_total_output"]
).fillna(0)

firm_summary["is_intermediary"] = (firm_summary["outsourcing_intensity"] > 0.9).astype(int)
firm_summary["is_outsourcing"]  = (firm_summary["outsourcing_intensity"] >= 0.01).astype(int)

firm_summary.to_stata("firm_year_summary.dta", write_index=False)

# ============================================================
# 确定主营产品（firm-year 内 total_output 最大，并列取 product_id 最小）
# ============================================================
df_sorted = df.sort_values(["year", "firm_id", "total_output", "product_id"],
                           ascending=[True, True, False, True])
main_product = (
    df_sorted
    .groupby(["year", "firm_id"], as_index=False)
    .first()[["year", "firm_id", "product_id", "total_output"]]
    .rename(columns={"product_id": "main_product", "total_output": "main_product_output"})
)

df = df.merge(main_product, on=["year", "firm_id"], how="left")
df["is_main"] = (df["product_id"] == df["main_product"]).astype(int)

# ============================================================
# 合并公司特征
# ============================================================
df = df.merge(firm_summary, on=["year", "firm_id"], how="left")

df["sales_percen"]        = df["total_output"] / df["firm_total_output"]
df["sales_relative_main"] = df["total_output"] / df["main_product_output"]

# ============================================================
# 合并相似度（对称矩阵，合并两个方向后去重）
# ============================================================
sim = pd.read_stata("full_product_similarity.dta")

sim_dir1 = sim.rename(columns={"product_1": "product_id", "product_2": "main_product"})
sim_dir2 = sim.rename(columns={"product_2": "product_id", "product_1": "main_product"})
sim_lookup = (
    pd.concat([sim_dir1, sim_dir2])
    .drop_duplicates(subset=["product_id", "main_product"])
    .reset_index(drop=True)
)

df = df.merge(sim_lookup, on=["product_id", "main_product"], how="left")

# 主营产品与自身相似度定义为 1
df.loc[df["is_main"] == 1, "input_similarity"]  = 1.0
df.loc[df["is_main"] == 1, "output_similarity"] = 1.0

# ============================================================
# 整理列顺序并保存
# ============================================================
cols = [
    "year", "firm_id", "product_id",
    "total_output", "outsourcing_value", "production_value",
    "outsourcing_percen", "sales_percen", "sales_relative_main",
    "is_main", "main_product", "main_product_output",
    "input_similarity", "output_similarity",
    "firm_total_output", "firm_total_outsource", "n_products",
    "outsourcing_intensity", "is_intermediary", "is_outsourcing",
]
df = df[cols].sort_values(["year", "firm_id", "total_output"],
                          ascending=[True, True, False]).reset_index(drop=True)

df.to_stata("full_data.dta", write_index=False)